## Objective

This notebook takes the raw Online Retail transaction dataset and produces a single, clean table that can be used without modification for customer-level analysis, CLV modeling, and BI reporting.

The dataset contains well-documented issues: missing customer identifiers, negative quantities, canceled invoices, zero-price items, and non-product stock codes. Any one of these, if ignored, will distort core customer metrics - inflating revenue, deflating retention rates, or misclassifying churn. This notebook identifies and resolves each issue against the business logic of a retail operation, rather than just the statistical distribution of the data.

The process follows six sequential steps:

1. Load the raw data and consolidate into a single working table.
2. Standardize column names, data types, and date formats to a consistent schema.
3. Remove records with no analytical value: canceled invoices, structurally invalid rows, and non-product stock codes.
4. Investigate the relationship between negative quantities and cancellations, and apply the appropriate treatment based on what the data shows.
5. Engineer a TotalPrice column (Quantity × Price) as the single source of truth for transaction-level revenue.
6. Export the cleaned table for direct loading into downstream notebooks and Power BI.

The output is the project's canonical transaction record - the single table that every CLV model, cohort analysis, and dashboard query reads from.

## Data Ingestion

The raw data is read from a single CSV file into a DataFrame, capturing every transaction recorded between December 2009 and December 2011. No filtering, aggregation, or transformation occurs at this stage. The goal is to establish a lossless starting point by preserving the complete transactional timeline exactly as provided.

Preserving the full two-year window matters directly for CLV work. Shortening or sampling the timeline would compromise time-based train/test splits, distort recency calculations, and weaken the model's ability to learn repeat-purchase patterns across realistic customer lifetimes. Every downstream step - feature engineering, cohort definition, holdout construction - relies on this full, unpruned history being intact from the start.

In [1]:
import pandas as pd
import numpy as np

input_path = "/kaggle/input/online-retail-ii-uci/online_retail_II.csv"

# Load without transformation to preserve the complete transactional record
raw_data = pd.read_csv(input_path)

print(f"Dataset shape: {raw_data.shape}")
raw_data.head()

Dataset shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Column Standardization and Data Type Enforcement

Column names and data types are standardized before applying any business logic. Skipping this step creates a fragile pipeline, one that works on the current file but breaks when a column is renamed or a date field is read as a string in a different export.

The process runs in four passes:

**First**, all column names are stripped of whitespace to eliminate invisible characters that cause silent mismatches.

**Second**, the schema is validated against a fixed list of expected columns: Invoice, StockCode, Description, Quantity, InvoiceDate, Price, CustomerID, and Country. Missing columns surface immediately rather than as KeyErrors downstream. Known schema variants are reconciled - InvoiceNo maps to Invoice, UnitPrice maps to Price.

**Third**, data types are enforced. InvoiceDate parses to datetime64 with unparseable values set to NaT. CustomerID casts to numeric with non-numeric values coerced to NaN. Both use `errors = 'coerce'` to preserve the row and flag the failure without crashing the pipeline.

**Fourth**, string columns are stripped of leading and trailing whitespace to prevent duplicate keys and fragmented groupings.

This section enforces the data contract at the boundary between raw input and analytical logic. All subsequent sections reference column names and types without defensive checks.

In [2]:
df = raw_data.copy()

# Remove invisible characters from column names
df.columns = [column.strip().replace(" ", "") for column in df.columns]

# Validate schema against expected columns
expected_columns = ["Invoice", "StockCode", "Description", "Quantity", 
                    "InvoiceDate", "Price", "CustomerID", "Country"]
missing_columns = [col for col in expected_columns if col not in df.columns]
print("Missing expected columns:", missing_columns)

# Reconcile schema variants across dataset versions
rename_map = {}
if "InvoiceNo" in df.columns:
    rename_map["InvoiceNo"] = "Invoice"
if "UnitPrice" in df.columns:
    rename_map["UnitPrice"] = "Price"
df = df.rename(columns = rename_map)

# Enforce data types with explicit coercion to preserve rows
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors = "coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors = "coerce")

# Strip whitespace from string fields to prevent fragmented groupings
for column in ["Invoice", "StockCode", "Description", "Country"]:
    if column in df.columns:
        df[column] = df[column].astype(str).str.strip()

df.info()

Missing expected columns: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1067371 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   CustomerID   824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


## Data Cleaning: Business-Reasonable Rules

Raw transaction data from retail systems carries structural noise: missing identifiers, canceled orders, returned goods, and pricing anomalies. Each distorts a different CLV lever - missing Customer IDs break the customer-level unit of analysis, cancellations inflate purchase frequency, and zero-price items drag down monetary metrics. The rules below are applied in sequence, each with a rationale tied to downstream CLV requirements.

**Missing identifiers.** Records lacking InvoiceDate, Invoice, or CustomerID are removed. These three fields form the minimum skeleton of a customer-level transaction: without them, the row cannot be assigned to a point in time, a purchase event, or an individual. Such rows may inform aggregate revenue totals but carry no signal for recency, frequency, or per-customer monetary value.

**Canceled invoices.** Invoices prefixed with "C" represent cancellations - orders initiated but never fulfilled. Including them would count purchase events that did not occur and revenue that was never recognized. They are removed to align the transaction table with completed commercial activity only.

**Returns and negative quantities.** The dataset contains negative Quantity values, which typically indicate returned items. The investigation below examines whether these returns exist independently of cancellations or are entirely contained within canceled invoices. The treatment depends on what the data shows.

**Invalid and extreme pricing.** Transactions with zero or negative Price are removed. These correspond to samples, administrative adjustments, or data entry errors - none of which represent revenue-generating activity. Extreme high-value outliers are filtered using percentile-based thresholds to prevent anomalous records from pulling CLV estimates away from the central behavior of the customer base.

In [3]:
clean_data = df.copy()

# Remove records lacking the minimum fields required for customer-level analysis
clean_data = clean_data.dropna(subset = ["InvoiceDate", "Invoice", "CustomerID"])

# Investigate the relationship between negative quantities and cancellations
# before applying removal rules
clean_data["is_cancelled"] = clean_data["Invoice"].astype(str).str.startswith("C")
clean_data["is_return"] = clean_data["Quantity"] < 0

print("Relationship between returns and cancellations:")
relationship_table = pd.crosstab(clean_data["is_cancelled"], 
                                  clean_data["is_return"], 
                                  margins = True)
print(relationship_table)
print("\nInterpretation:")
print(f"Total negative-quantity rows: {clean_data['is_return'].sum():,}")
print(f"Negative-quantity rows on cancelled invoices: {(clean_data['is_return'] & clean_data['is_cancelled']).sum():,}")
print(f"Negative-quantity rows on non-cancelled invoices: {(clean_data['is_return'] & ~clean_data['is_cancelled']).sum():,}")

Relationship between returns and cancellations:
is_return      False   True     All
is_cancelled                       
False         805620      0  805620
True               0  18744   18744
All           805620  18744  824364

Interpretation:
Total negative-quantity rows: 18,744
Negative-quantity rows on cancelled invoices: 18,744
Negative-quantity rows on non-cancelled invoices: 0


### Returns Investigation: Empirical Finding

The cross-tabulation reveals a structural characteristic of this dataset: **every negative-quantity transaction is associated with a canceled invoice** (invoice number prefixed with "C"). There are zero negative-quantity rows on non-cancelled invoices.

This means the dataset does not contain genuine product returns as a separate economic event. What appears as a return is actually the reversal of an unfulfilled order, the negative quantity cancels out the positive quantity on the paired cancellation invoice. These are administrative reversals, not customer returns.

**Implication for CLV:** When canceled invoices are removed, all negative-quantity rows are removed with them. The cleaned dataset contains only completed, positive-quantity transactions. This is correct for the modeling objective: CLV should estimate value from completed purchases, and a canceled order generates no customer value.

If the dataset did contain genuine returns (negative quantities on non-cancelled invoices), they would be retained as negative revenue contributions. That condition does not exist in this data, and the cleaning logic reflects that empirically.

In [4]:
# Remove cancellations, which also removes all negative-quantity rows
clean_data = clean_data.loc[~clean_data["is_cancelled"]].copy()
clean_data = clean_data.drop(columns = ["is_cancelled", "is_return"])

# Enforce numeric types for quantity and price calculations
clean_data["Quantity"] = pd.to_numeric(clean_data["Quantity"], errors = "coerce")
clean_data["Price"] = pd.to_numeric(clean_data["Price"], errors = "coerce")

# Remove non-revenue transactions (zero or negative prices)
clean_data = clean_data.loc[clean_data["Price"] > 0].copy()

# Calculate transaction-level revenue from completed purchases only
clean_data["TotalPrice"] = clean_data["Quantity"] * clean_data["Price"]

# Remove zero-quantity rows that carry no economic signal
clean_data = clean_data.loc[clean_data["Quantity"] != 0].copy()

# Remove extreme outliers using percentile thresholds
# Thresholds applied to positive quantities only, as no returns remain
purchase_q_threshold = clean_data["Quantity"].quantile(0.999)
purchase_price_threshold = clean_data["Price"].quantile(0.999)

clean_data = clean_data.loc[
    (clean_data["Quantity"] <= purchase_q_threshold) & 
    (clean_data["Price"] <= purchase_price_threshold)
].copy()

print(f"Rows after cleaning: {len(clean_data):,}")
print(f"Negative quantities remaining: {(clean_data['Quantity'] < 0).sum():,}")
print(f"Final return rate: {(clean_data['Quantity'] < 0).mean():.4%}")

Rows after cleaning: 804,008
Negative quantities remaining: 0
Final return rate: 0.0000%


### Post-Cleaning Data Quality Summary

After cleaning rules are applied, a structured summary confirms the transformations achieved their intended outcomes without excessive data loss.

The summary reports on seven dimensions, each checked against the pre-cleaning baseline:

- **Row count** (raw vs. clean) to quantify total data reduction.
- **Unique customers** to verify customer-level coverage remains intact.
- **Date range** to confirm no entire time period was inadvertently filtered.
- **Country count** to ensure geographic breadth is preserved.
- **Return transaction proportion** to validate that no unexpected negative quantities remain.

A drop in unique customers exceeding the proportion of rows removed would signal that customer ID handling needs revisiting. A collapsed date range would indicate a filter caught more than canceled invoices. A non-zero return proportion after cancellation removal would indicate that genuine returns exist and require different treatment, this is not the case in this dataset.

In [5]:
data_summary = {
    "rows_raw": len(df),
    "rows_clean": len(clean_data),
    "rows_removed_pct": (1 - len(clean_data) / len(df)) * 100,
    "unique_customers": clean_data["CustomerID"].nunique(),
    "min_date": clean_data["InvoiceDate"].min(),
    "max_date": clean_data["InvoiceDate"].max(),
    "countries": clean_data["Country"].nunique() if "Country" in clean_data.columns else None,
    "negative_quantity_rows": (clean_data["Quantity"] < 0).sum(),
    "pct_negative_quantity": (clean_data["Quantity"] < 0).mean() * 100
}

pd.Series(data_summary).round(4)

rows_raw                              1067371
rows_clean                             804008
rows_removed_pct                    24.673989
unique_customers                         5840
min_date                  2009-12-01 07:45:00
max_date                  2011-12-09 12:50:00
countries                                  41
negative_quantity_rows                      0
pct_negative_quantity                     0.0
dtype: object

## Data Quality Discussion

The cleaning sequence produces a transaction table consisting entirely of positive-quantity, revenue-generating purchases. This outcome reflects the structure of the source data: all negative-quantity rows are tied to canceled invoices, and removing cancellations removes the negative quantities with them.

This is the correct outcome for CLV modeling as the model estimates lifetime value from completed, economically meaningful transactions. A canceled order generates no customer value, and its reversal carries no signal for future purchase behavior. 

The dataset does not contain genuine product returns as a separate event type. If it is updated at a later stage and provides data that distinguishes genuine returns from administrative reversals, the cleaning pipeline can be extended to retain those returns as negative revenue contributions.

## Persisting the Clean Transaction Dataset

The cleaned transaction dataset is saved as a CSV to create a stable, reusable foundation for the remainder of the project. All downstream notebooks and BI dashboards read from this single source, rather than reapplying cleaning logic independently.

This separation between raw ingestion and cleaned output improves reproducibility, simplifies debugging, and supports integration with Power BI and subsequent modeling workflows.

In [6]:
file_name = "transactions_clean.csv"

clean_data.to_csv(file_name, index = False)
print(f"Cleaned transactions saved to: {file_name}")
print(f"Final shape: {clean_data.shape}")

clean_data.head()

Cleaned transactions saved to: transactions_clean.csv
Final shape: (804008, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0
